**Imports & Setup**

This cell imports all the necessary Python libraries required for the project, including Pandas for data manipulation, Psycopg2 for PostgreSQL database connectivity, and SQLAlchemy for managing the database engine.

In [12]:
# Importing libraries
import pandas as pd
import psycopg2
from sqlalchemy import create_engine, URL

**Database Connection**

This cell establishes a connection to the PostgreSQL database (`insurance_bitemporal`) using SQLAlchemy. It creates an engine with the necessary credentials and assigns it to `myconn`, which can be used for executing SQL queries and fetching data into Pandas DataFrames.

In [13]:
# Create the URL object which safely encodes passwords with special characters
url_object = URL.create(
    drivername="postgresql+psycopg2",
    username="postgres",
    password="#Ssr@0712#",  # Your plain password here
    host="localhost",
    port=5432,
    database="insurance_bitemporal",
)

# Create the engine using the safe URL
engine = create_engine(
    url_object,
    connect_args={'options': '-csearch_path=insurance'}
)

# Assign it to myconn so your existing pd.read_sql calls work
myconn = engine

**Database Initialization**

This cell defines a function that automatically executes the Phase 2 DDL and DML SQL scripts to create the necessary tables and insert initial data if they are missing. This ensures the Bi-Temporal database is set up and ready for queries.

In [14]:
def initialize_database(engine):
    """
    Checks if the database tables exist and are populated.
    If not, executes the DDL, Functions, Procedures, Triggers,
    and DML (baseline data). 
    
    The Temporal Demo is NOT run automatically here.
    """
    raw_conn = engine.raw_connection()
    try:
        with raw_conn.cursor() as cursor:
            # Check if a core table ('company_branch') exists in the 'insurance' schema
            cursor.execute("SELECT EXISTS (SELECT FROM pg_tables WHERE schemaname = 'insurance' AND tablename = 'company_branch');")
            exists = cursor.fetchone()[0]
            
            if not exists:
                print("Tables not found. Initializing database with Phase 2 scripts...")
                
                # Execute DDL
                with open(r"f:\Projects\DBMS\Phase 2-Bi-Temporal_DBMS\Insurance database Bi-Temporal DDL.sql", "r", encoding="utf-8") as ddl_file:
                    cursor.execute(ddl_file.read())
                
                # Execute Functions
                with open(r"f:\Projects\DBMS\Phase 2-Bi-Temporal_DBMS\Insurance database Functions.sql", "r", encoding="utf-8") as func_file:
                    cursor.execute(func_file.read())
                
                # Execute Procedures
                with open(r"f:\Projects\DBMS\Phase 2-Bi-Temporal_DBMS\Insurance database Procedures.sql", "r", encoding="utf-8") as proc_file:
                    cursor.execute(proc_file.read())
                
                # Execute Triggers
                with open(r"f:\Projects\DBMS\Phase 2-Bi-Temporal_DBMS\Insurance database Triggers.sql", "r", encoding="utf-8") as trig_file:
                    cursor.execute(trig_file.read())
                    
                # Execute DML (baseline data: branches, agents, customers, policies, etc.)
                with open(r"f:\Projects\DBMS\Phase 2-Bi-Temporal_DBMS\Insurance database Bi-Temporal DML.sql", "r", encoding="utf-8") as dml_file:
                    cursor.execute(dml_file.read())
                    
                raw_conn.commit()
                print("Database tables and baseline data initialized successfully.")
            else:
                # Tables exist, check if data exists
                cursor.execute("SELECT COUNT(*) FROM insurance.company_branch;")
                count = cursor.fetchone()[0]
                if count == 0:
                    print("Tables exist but no data found. Inserting baseline data...")
                    with open(r"f:\Projects\DBMS\Phase 2-Bi-Temporal_DBMS\Insurance database Bi-Temporal DML.sql", "r", encoding="utf-8") as dml_file:
                        cursor.execute(dml_file.read())
                    raw_conn.commit()
                    print("Baseline data inserted successfully.")
                else:
                    print("Database already initialized and contains baseline data.")
    except Exception as e:
        raw_conn.rollback()
        print(f"Error during initialization: {e}")
    finally:
        raw_conn.close()

# Automatically initialize database when running this notebook
initialize_database(engine)


Database already initialized and contains data.


### Run Bi-Temporal Demonstrations (Optional)
The cell below executes the `Insurance database Temporal Demo.sql` file. This script runs a series of simulated real-world updates (address changes, policy renewals, etc.).\n
When this runs, the triggers we set up will automatically archive the old state into the `_history` tables before applying the new state.

In [ ]:
def run_temporal_demo(engine):
    """
    Optional function to run the temporal demonstration script.
    This simulates time-travel events like address changes and renewals.
    """
    raw_conn = engine.raw_connection()
    try:
        with raw_conn.cursor() as cursor:
            print("Executing Temporal Demo...")
            # Execute Temporal Demo (bi-temporal lifecycle operations)
            with open(r"f:\Projects\DBMS\Phase 2-Bi-Temporal_DBMS\Insurance database Temporal Demo.sql", "r", encoding="utf-8") as demo_file:
                cursor.execute(demo_file.read())
            
            raw_conn.commit()
            print("Temporal demonstrations executed! History tables are now populated.")
    except Exception as e:
        raw_conn.rollback()
        print(f"Error during temporal demo execution: {e}")
    finally:
        raw_conn.close()

# Uncomment the line below to run the temporal demonstrations:
# run_temporal_demo(engine)
